# Meta data from smr file

In [3]:
import pandas as pd
import re

# Path to the .smr file
summary_file = r"C:\Users\knand\OneDrive\Desktop\preterm\term-preterm-ehg-database-1.0.1\tpehgdb.smr"

def load_smr_file(file_path):
    """
    Parse the .smr file to load its metadata and numerical features.

    Args:
        file_path (str): Path to the .smr file.
    
    Returns:
        metadata_df (pd.DataFrame): DataFrame containing metadata.
        features_df (pd.DataFrame): DataFrame containing numerical features.
    """
    metadata = []
    features = []
    is_feature_section = False
    
    with open(file_path, "r") as file:
        for line in file:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
            
            # Identify the feature section (purely numeric lines)
            if re.match(r"^\d+\.?\d*(\s+\d+\.?\d*)*$", line):
                is_feature_section = True
            
            if not is_feature_section:
                # Process metadata lines with "|"
                if "|" in line and not line.startswith("-"):
                    row = [col.strip() for col in line.split("|")]
                    metadata.append(row)
            else:
                # Process numerical feature lines
                features.append([float(x) for x in line.split() if x.strip()])
    
    # Extract column names from metadata
    metadata_columns = metadata[0] if metadata else []
    metadata_data = metadata[1:] if len(metadata) > 1 else []
    
    # Create DataFrames
    metadata_df = pd.DataFrame(metadata_data, columns=metadata_columns)
    features_df = pd.DataFrame(features)
    
    return metadata_df, features_df

# Load the .smr file
metadata_df, features_df = load_smr_file(summary_file)

# Display the DataFrames
print("Metadata:")
print(metadata_df.head())
print("\nFeatures:")
print(features_df.head())


Metadata:
      Record Gestation Rec. time      Group Premature Early
0  tpehg1007     35.00     31.29   >=26-PRE         t     f
1  tpehg1021     38.57     22.29   <26-TERM         f     t
2  tpehg1022     38.57     31.00  >=26-TERM         f     f
3  tpehg1027     37.14     31.29  >=26-TERM         f     f
4  tpehg1029     38.57     31.00  >=26-TERM         f     f

Features:
Empty DataFrame
Columns: []
Index: []


In [7]:
output_file = "metadata.csv"
metadata_df.to_csv(output_file, index=False)

print(f"Metadata saved to {output_file}")

Metadata saved to metadata.csv


# clinical data from hea files

In [2]:
import os
import pandas as pd
import pandas as pd
import re
# Function to process a .hea file and extract clinical features
def process_header_file(file):
    names = []
    values = []

    try:
        with open(file, 'r') as ifp:
            lines = ifp.readlines()
            # Find the start of the clinical data
            start_idx = next((i for i, line in enumerate(lines) if line.startswith('#')), len(lines))

            # Extract clinical features from the header
            for line in lines[start_idx + 1:]:
                parts = line.strip().split(maxsplit=2)
                if len(parts) >= 3:
                    _, name, value = parts
                    names.append(name)
                    values.append(value)
                else:
                    print(f"Warning: Skipping malformed line in {file}: {line.strip()}")
    except Exception as e:
        print(f"Error processing {file}: {e}")

    return names, values

In [3]:
# Directory containing .hea files
hea_dir = r'C:\Users\knand\OneDrive\Desktop\preterm\PTB\Dataset\tocogram'

# Collect existing .hea files
existing_files = {f.split('.')[0] for f in os.listdir(hea_dir) if f.endswith('.hea')}

# Initialize a list to store clinical feature vectors
vectors = []
column_names = ['Record']  # First column for the record identifier

# Process each .hea file
for file in existing_files:
    print(f"Processing file: {file}.hea")
    try:
        file_path = os.path.join(hea_dir, f"{file}.hea")
        names, values = process_header_file(file_path)

        # Ensure the column names are consistent across files
        if len(vectors) == 0:  # First file sets the column names
            column_names.extend(names)
        elif names != column_names[1:]:
            print(f"Warning: Column names mismatch in {file}.hea")
            continue

        # Append the record's clinical data
        vectors.append([file] + values)

    except Exception as e:
        print(f"Error processing {file}: {e}")

# Create a DataFrame and save to CSV
clinical_df = pd.DataFrame(vectors, columns=column_names)
output_csv_path = "clinical_features.csv"
clinical_df.to_csv(output_csv_path, index=False)

print(f"Clinical features saved to {output_csv_path}")


Processing file: tpehgt_n004.hea
Processing file: tpehgt_p010.hea
Processing file: tpehgt_t012.hea
Processing file: tpehgt_p007.hea
Processing file: tpehgt_p011.hea
Processing file: tpehgt_t009.hea
Processing file: tpehgt_t013.hea
Processing file: tpehgt_p003.hea
Processing file: tpehgt_n002.hea
Processing file: tpehgt_t008.hea
Processing file: tpehgt_t005.hea
Processing file: tpehgt_p005.hea
Processing file: tpehgt_t011.hea
Processing file: tpehgt_t001.hea
Processing file: tpehgt_n003.hea
Processing file: tpehgt_p008.hea
Processing file: tpehgt_n001.hea
Processing file: tpehgt_p002.hea
Processing file: tpehgt_t004.hea
Processing file: tpehgt_t006.hea
Processing file: tpehgt_p009.hea
Processing file: tpehgt_p012.hea
Processing file: tpehgt_p004.hea
Processing file: tpehgt_t010.hea
Processing file: tpehgt_t007.hea
Processing file: tpehgt_p006.hea
Processing file: tpehgt_n005.hea
Processing file: tpehgt_t002.hea
Processing file: tpehgt_t003.hea
Processing file: tpehgt_p001.hea
Processing

## cleaning

In [4]:
clinical_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Record              31 non-null     object
 1   RecID               31 non-null     object
 2   RecType             31 non-null     object
 3   Gestation           31 non-null     object
 4   Rectime             31 non-null     object
 5   Age                 31 non-null     object
 6   Parity              31 non-null     object
 7   Abortions           31 non-null     object
 8   Weight              31 non-null     object
 9   Placental_position  31 non-null     object
 10  Smoker              31 non-null     object
dtypes: object(11)
memory usage: 2.8+ KB


In [5]:
# Replace common placeholders with NaN
import numpy as np
clinical_df.replace(["", "NA", "None", "null"], np.nan, inplace=True)

# Check for missing values
print(clinical_df.isnull().sum())


Record                0
RecID                 0
RecType               0
Gestation             0
Rectime               0
Age                   5
Parity                5
Abortions             5
Weight                5
Placental_position    0
Smoker                5
dtype: int64


In [6]:
missing_percentage = clinical_df.isnull().mean() * 100
print(missing_percentage)


Record                 0.000000
RecID                  0.000000
RecType                0.000000
Gestation              0.000000
Rectime                0.000000
Age                   16.129032
Parity                16.129032
Abortions             16.129032
Weight                16.129032
Placental_position     0.000000
Smoker                16.129032
dtype: float64


In [7]:
# Ensure numerical columns contain only numeric data
numerical_cols = ['Age', 'Parity', 'Abortions', 'Weight']

for col in numerical_cols:
    clinical_df[col] = pd.to_numeric(clinical_df[col], errors='coerce')  # Convert to numeric, invalid values become NaN

# Fill missing values with the median
clinical_df[numerical_cols] = clinical_df[numerical_cols].fillna(clinical_df[numerical_cols].median())


In [11]:
# Categorical columns: Fill missing values with the mode
categorical_cols = ['Placental_position', 'Smoker']
clinical_df[categorical_cols] = clinical_df[categorical_cols].fillna(clinical_df[categorical_cols].mode().iloc[0])


In [12]:
# Categorical columns: Mapping categories to numeric values
categorical_map = {
    
    'Placental_position': {'front': 0, 'end': 1},
    'Smoker': {'no': 0, 'yes': 1}
}

# Apply the mappings to the categorical columns
for col in categorical_cols:
    clinical_df[col] = clinical_df[col].map(categorical_map[col])

# Check if the encoding worked correctly
print(clinical_df[categorical_cols].head())


   Placental_position  Smoker
0                 NaN       0
1                 0.0       1
2                 1.0       0
3                 0.0       1
4                 0.0       1


In [13]:
output = "clinical_features_cleaned.csv"
clinical_df.to_csv(output, index=False)

print(f"Clinical features saved to {output}")


Clinical features saved to clinical_features_cleaned.csv


In [31]:
import pandas as pd

# Merge the DataFrames on the 'Record' column
merged_df = pd.merge(metadata_df, clinical_df, on='Record', how='inner')

# Check the first few rows of the merged DataFrame
print(merged_df.head())

# Save the merged DataFrame to a new CSV file
merged_df.to_csv('merged_clinical_data.csv', index=False)


      Record Gestation_x Rec. time      Group Premature Early RecID  \
0  tpehg1007       35.00     31.29   >=26-PRE         t     f  1007   
1  tpehg1021       38.57     22.29   <26-TERM         f     t  1021   
2  tpehg1022       38.57     31.00  >=26-TERM         f     f  1022   
3  tpehg1027       37.14     31.29  >=26-TERM         f     f  1027   
4  tpehg1029       38.57     31.00  >=26-TERM         f     f  1029   

  Gestation_y Rectime   Age  Parity  Abortions  Weight  Hypertension  \
0          35    31.3  30.0     0.0        0.0    58.0             0   
1        38.6    22.3  29.0     0.0        0.0    63.0             0   
2        38.6      31  29.0     0.0        0.0    70.0             0   
3        37.1    31.3  27.0     0.0        1.0   100.0             0   
4        38.6      31  28.0     0.0        2.0    72.0             0   

   Diabetes  Placental_position  Bleeding_first_trimester  \
0         0                   0                         0   
1         0       

In [15]:
import pandas as pd

# Load the CSV file
file_path = "clinical_features_cleaned.csv"  # Update with your actual file path
clinical_data = pd.read_csv(file_path)


# Encode 'Premature' and 'Early' columns
clinical_data['RecType'] = clinical_data['RecType'].map({'Preterm': 1, 'Term': 0})

# Save the processed file
output_file_path = "final_clinical.csv"
clinical_data.to_csv(output_file_path, index=False)

print(f"Processed clinical data saved to {output_file_path}")


Processed clinical data saved to final_clinical.csv


In [2]:
import pandas as pd

# Load the individual CSV files
time_domain_features = pd.read_csv("time_domain_features.csv")
frequency_domain_features = pd.read_csv("frequency_domain_features.csv")
time_frequency_features = pd.read_csv("time_frequency_features.csv")

# Merge the files on 'Record' and 'Channel'
merged_data = pd.merge(time_domain_features, frequency_domain_features, on=['Record', 'Channel'], how='inner')
merged_data = pd.merge(merged_data, time_frequency_features, on=['Record', 'Channel'], how='inner')

# Save the merged data to a single CSV file
output_file = "combined_features.csv"
merged_data.to_csv(output_file, index=False)

print(f"Combined features saved to {output_file}")


Combined features saved to combined_features.csv


In [20]:
import pandas as pd

# Load the feature and clinical files
combined_features = pd.read_csv("final_features.csv")
final_clinical = pd.read_csv("cleaned_file.csv")

# Repeat rows in clinical data to match the 3 channels for each Record in features data
final_clinical_expanded = final_clinical.loc[final_clinical.index.repeat(3)].reset_index(drop=True)

# Add Channel column to match the structure of combined_features
final_clinical_expanded['Channel'] = [1, 2, 3] * (len(final_clinical) // 1)

# Merge the expanded clinical data with the combined features
merged_data = pd.merge(combined_features, final_clinical_expanded, on=['Record', 'Channel'], how='inner')

# Save the final merged data to a CSV file
output_file = "merged_features.csv"
merged_data.to_csv(output_file, index=False)

print(f"Merged data saved to {output_file}")


Merged data saved to merged_features.csv


In [8]:
import pandas as pd

# Load the merged data
merged_file = "merged_features_clinical.csv"
merged_data = pd.read_csv(merged_file)

# Drop the 'Record' column
merged_data = merged_data.drop(columns=['Record'])

merged_data = merged_data.drop(columns=['Gestation_y'])
merged_data = merged_data.drop(columns=['Rectime'])
merged_data = merged_data.rename(columns={'Rec. time': 'RecTime'})

merged_data = merged_data.rename(columns={'Gestation_x': 'Gestation'})

# Rearrange columns in the specified order
column_order = [
    'Channel', 'Mean', 'Variance', 'Skewness', 'Kurtosis', 'RMS', 'IQR', 'Zero Crossings', 'Envelope',
    'PSD Band 0.3–0.6 Hz', 'PSD Band 0.6–1.0 Hz', 'Dominant Frequency', 'Mean Frequency', 'Total Power', 
    'Spectral Entropy', 'Wavelet Energy Level 1', 'Wavelet Energy Level 2', 'Wavelet Energy Level 3', 
    'Wavelet Energy Level 4', 'Wavelet Energy Level 5', 'Wavelet Entropy', 'Instantaneous Amplitude Mean', 
    'Instantaneous Amplitude Std', 'Instantaneous Frequency Mean', 'Instantaneous Frequency Std', 
    'Gestation', 'RecTime', 'RecID', 'Age', 'Parity', 
    'Abortions', 'Weight', 'Hypertension', 'Diabetes', 'Placental_position', 'Bleeding_first_trimester', 
    'Bleeding_second_trimester', 'Funneling', 'Smoker', 'Premature', 'Early'
]

# Ensure the 'RecID' column is first
merged_data = merged_data[['RecID'] + [col for col in column_order if col != 'RecID']]

# Save the final adjusted file
output_file = "Final_Features.csv"
merged_data.to_csv(output_file, index=False)

print(f"Rearranged data saved to {output_file}")


Rearranged data saved to Final_Features.csv


In [18]:
import pandas as pd  

# Load the CSV file  
df = pd.read_csv("final_clinical.csv")  
df = df.drop(columns=["RecID"], errors="ignore")

# Drop columns starting with "tpehgt_n" and the "RecID" column  
df = df.loc[:, ~df.columns.str.startswith("tpehgt_n")]  
  # Ignore error if "RecID" doesn't exist  

# Save the cleaned CSV  
df.to_csv("cleaned_clinical_file.csv", index=False)  

print("Columns removed and file saved successfully.")  


Columns removed and file saved successfully.


In [19]:
df = df.rename(columns={"RecType": "Premature"})  

# Save the cleaned CSV  
df.to_csv("cleaned_file.csv", index=False)  
